# YOLO: классы мобильных роботов и разметка датасета

Готовая COCO-модель YOLO не знает учебные классы вида
`tracked_robot`, `legged_robot`, `wheeled_robot`.
Поэтому для пособия нужен не пример с автобусом, а пример подготовки
собственного YOLO-датасета по мобильным роботам.

Этот ноутбук создает файл разметки YOLO для фотографии из пособия и
визуализирует классы: колесные, гусеничные и шагающие роботы.


In [ ]:
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import yaml

cwd = Path.cwd()
MATERIALS = cwd / "verified_materials" if (cwd / "verified_materials").exists() else cwd
ASSETS = MATERIALS / "assets"
OUTPUTS = MATERIALS / "outputs"
YOLO_DATASET = MATERIALS / "robot_yolo_dataset"
OUTPUTS.mkdir(parents=True, exist_ok=True)

image_path = ASSETS / "mobile_robots_lab.jpg"
img = cv2.imread(str(image_path))
if img is None:
    raise FileNotFoundError(image_path)

height, width = img.shape[:2]
print("Image:", image_path)
print("Shape:", width, height)


In [ ]:
class_names = ["wheeled_robot", "tracked_robot", "legged_robot"]

# Coordinates are x1, y1, x2, y2 in pixels on mobile_robots_lab.jpg.
annotations = [
    ("legged_robot", 55, 320, 326, 498),
    ("wheeled_robot", 172, 258, 245, 311),
    ("wheeled_robot", 494, 235, 568, 316),
    ("wheeled_robot", 590, 235, 674, 327),
    ("wheeled_robot", 673, 265, 759, 348),
    ("wheeled_robot", 724, 342, 824, 452),
    ("tracked_robot", 566, 389, 685, 472),
    ("tracked_robot", 478, 430, 556, 525),
]

colors = {
    "wheeled_robot": (255, 180, 0),
    "tracked_robot": (20, 160, 255),
    "legged_robot": (0, 220, 80),
}

label_text = {
    "wheeled_robot": "wheeled",
    "tracked_robot": "tracked",
    "legged_robot": "legged",
}


In [ ]:
dataset_image_dir = YOLO_DATASET / "images" / "train"
dataset_label_dir = YOLO_DATASET / "labels" / "train"
dataset_image_dir.mkdir(parents=True, exist_ok=True)
dataset_label_dir.mkdir(parents=True, exist_ok=True)

copied_image = dataset_image_dir / image_path.name
copied_image.write_bytes(image_path.read_bytes())

label_lines = []
for label, x1, y1, x2, y2 in annotations:
    class_id = class_names.index(label)
    x_center = ((x1 + x2) / 2) / width
    y_center = ((y1 + y2) / 2) / height
    box_width = (x2 - x1) / width
    box_height = (y2 - y1) / height
    label_lines.append(
        f"{class_id} {x_center:.6f} {y_center:.6f} {box_width:.6f} {box_height:.6f}"
    )

label_path = dataset_label_dir / "mobile_robots_lab.txt"
label_path.write_text("\n".join(label_lines) + "\n", encoding="utf-8")

data_yaml = {
    "path": str(YOLO_DATASET.resolve()),
    "train": "images/train",
    "val": "images/train",
    "names": {i: name for i, name in enumerate(class_names)},
}
data_yaml_path = YOLO_DATASET / "mobile_robots.yaml"
data_yaml_path.write_text(yaml.safe_dump(data_yaml, sort_keys=False, allow_unicode=True), encoding="utf-8")

print("YOLO label file:", label_path)
print("YOLO data file:", data_yaml_path)
print(label_path.read_text(encoding="utf-8"))


In [ ]:
annotated = img.copy()

for i, (label, x1, y1, x2, y2) in enumerate(annotations, start=1):
    color = colors[label]
    cv2.rectangle(annotated, (x1, y1), (x2, y2), color, 3)
    text = f"{i}: {label_text[label]}"
    cv2.putText(
        annotated,
        text,
        (x1, max(25, y1 - 8)),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.62,
        color,
        2,
        cv2.LINE_AA,
    )

out_path = OUTPUTS / "yolo_robot_classes_annotated.jpg"
cv2.imwrite(str(out_path), annotated)
print("Saved:", out_path)

plt.figure(figsize=(12, 7))
plt.imshow(cv2.cvtColor(annotated, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("Разметка мобильных роботов для YOLO")
plt.show()


## Как обучать YOLO после расширения датасета

Мини-пример выше нужен для пособия и проверки формата.
Для настоящей классификации/детекции нужно собрать больше изображений
каждого класса и разметить их в том же YOLO-формате.

После расширения датасета можно запустить:

```python
from ultralytics import YOLO

model = YOLO("verified_materials/models/yolov8n.pt")
model.train(
    data="verified_materials/robot_yolo_dataset/mobile_robots.yaml",
    epochs=50,
    imgsz=640,
    device="cpu",
    project="verified_materials/runs",
    name="mobile_robot_detector",
)
```
